#  Automated Data Quality and Fraud Alerts

Validate Silver transactions for missing IDs, invalid amounts,
invalid timestamps, and duplicate transaction IDs.

Generate alerts for transactions flagged as fraudulent or
classified as HIGH_VALUE or UNUSUAL_LOCATION.

Input: Silver Delta
Output: Quality summary and fraud alerts# Automated Data Quality and Fraud Alerts

Validate Silver transactions for missing IDs, invalid amounts,
invalid timestamps, and duplicate transaction IDs.

Generate alerts for transactions flagged as fraudulent or
classified as HIGH_VALUE or UNUSUAL_LOCATION.

Input: Silver Delta
Output: Quality summary and fraud alertsmated Data Quality and Fraud Alerts

Validate Silver transactions for missing IDs, invalid amounts,
invalid timestamps, and duplicate transaction IDs.

Generate alerts for transactions flagged as fraudulent or
classified as HIGH_VALUE or UNUSUAL_LOCATION.

Input: Silver Delta
Output: Quality summary and fraud alerts

### Python: Data-quality checks

In [0]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, when
)

silver_path = (
    "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

silver_df = spark.read.format("delta").load(silver_path)

checks = silver_df.agg(
    count("*").alias("total_records"),
    spark_sum(
        when(col("transaction_id").isNull(), 1).otherwise(0)
    ).alias("missing_transaction_id"),
    spark_sum(
        when(col("account_id").isNull(), 1).otherwise(0)
    ).alias("missing_account_id"),
    spark_sum(
        when(col("amount").isNull() | (col("amount") <= 0), 1)
        .otherwise(0)
    ).alias("invalid_amount"),
    spark_sum(
        when(col("transaction_timestamp").isNull(), 1).otherwise(0)
    ).alias("invalid_timestamp")
)

display(checks)

print("Data-quality checks completed.")

total_records,missing_transaction_id,missing_account_id,invalid_amount,invalid_timestamp
1296,0,0,0,0


Data-quality checks completed.


### Python: Generate fraud alerts

In [0]:
from pyspark.sql.functions import current_timestamp

alerts_path = (
    "abfss://gold@stbankingfrauddetection.dfs.core.windows.net/"
    "fraud_alerts/"
)

fraud_alerts = (
    silver_df
    .filter(
        (col("is_fraud") == True)
        | col("transaction_type").isin(
            "HIGH_VALUE", "UNUSUAL_LOCATION"
        )
    )
    .select(
        "transaction_id",
        "account_id",
        "amount",
        "merchant",
        "location",
        "transaction_type",
        "is_fraud"
    )
    .withColumn("alert_generated_at", current_timestamp())
)

display(fraud_alerts.limit(20))

print("Fraud alerts generated:", fraud_alerts.count())

transaction_id,account_id,amount,merchant,location,transaction_type,is_fraud,alert_generated_at
2a0efe1c-a8b7-4fe3-a7c4-9d75d12aa796,ACC1008,121428.85,Fuel Station,Chennai,HIGH_VALUE,true,2026-10-20T14:31:50.386394Z
44e5b3ab-f7f1-4760-bc31-f973cab09248,ACC1003,2535.75,Restaurant,Mumbai,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z
8f1fc952-2b0e-4522-825c-e2e7fe2a4397,ACC1013,89736.61,Fuel Station,Chennai,HIGH_VALUE,true,2026-10-20T14:31:50.386394Z
f145a0c4-7d59-4cd5-b27e-b184c928071c,ACC1016,9540.95,Restaurant,Mumbai,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z
90c8aba3-3a0b-4d43-a93d-739d70c2d458,ACC1014,8884.03,Electronics Store,Bengaluru,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z
e4971547-37cb-401b-934e-9b09567dce55,ACC1020,6020.49,Restaurant,Hyderabad,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z
0777df0c-e7c7-446a-8d76-8737cb621549,ACC1007,133584.77,Electronics Store,Chennai,HIGH_VALUE,true,2026-10-20T14:31:50.386394Z
f1bc4431-2942-41c9-99ae-bf3c2538cd2a,ACC1006,89247.3,Online Shopping,Chennai,HIGH_VALUE,true,2026-10-20T14:31:50.386394Z
ee554589-707d-4f6d-9d74-10855f31c5ea,ACC1002,4770.02,Grocery Store,Mumbai,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z
5f94b21b-9594-4639-b2e1-36b1a4f7e99b,ACC1018,8395.16,Fuel Station,Mumbai,UNUSUAL_LOCATION,true,2026-10-20T14:31:50.386394Z


Fraud alerts generated: 144


### Save the alerts

In [0]:
(
    fraud_alerts.write
    .format("delta")
    .mode("overwrite")
    .save(alerts_path)
)

print("Fraud alerts saved to ADLS Gen2.")

Fraud alerts saved to ADLS Gen2.
